# 07.07 · Custom agentic RAG and SQL agents in LangGraph

**Module 07 · LangGraph Fundamentals** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 90–120 min |
| 📋 Prerequisites | Lessons 07.02–07.04; Module 06 (corpus, BM25, cited answers). |
| 🔑 Keys | none (optional: `GROQ_API_KEY`) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Build** a custom RAG graph: `retrieve` → `grade_documents` → (`rewrite` → `retrieve`) → `answer`, with a bounded retry.
2. **Build** a SQL graph: `list_tables` → `get_schema` → `generate_query` → `check_query` → `run_query`, **read-only**, with an error-fix loop.
3. **Compose** both as subgraphs under a router and read the combined diagram with `xray=True`.
4. **Explain** why each step is a node: you can test it, time it and see it in Studio.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Custom RAG agent](https://docs.langchain.com/oss/python/langgraph/agentic-rag) | The docs' version: `grade_documents`, `rewrite_question`, `generate_answer` |
| [Custom SQL agent](https://docs.langchain.com/oss/python/langgraph/sql-agent) | List tables → schema → generate → **check** → run; why to check before running |
| [Use subgraphs](https://docs.langchain.com/oss/python/langgraph/use-subgraphs) | Adding a compiled graph as a node; shared keys; `xray` |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langgraph", "langchain-core",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/07-langgraph-fundamentals/).
import importlib.util
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Step 4: Silence known, harmless noise so real problems stand out.
import os
import warnings

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (07.07 reuses Module 06): no usage telemetry.
# Same as Lesson 02.04: langchain-core 1.6.9/1.6.10 Pydantic serializer warning about the injected runtime.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

In [ ]:
# 🧪 Offline: the Module 06 corpus, chunked, with BM25 (no embeddings needed for this lesson).
from mastery.graphs import show_graph
from mastery.rag import BM25, load_corpus, split_documents

chunks = split_documents(load_corpus(REPO_ROOT / "data" / "samples" / "rag" / "corpus"), "markdown", 800, 100)
bm25 = BM25(chunks)
print(len(chunks), "chunks")

**You should now see** the chunk count from Module 06 (a few dozen). BM25 is enough here: the lesson is about the **graph** around retrieval.

---
## Part 1 · A custom RAG graph

Module 06's chain always answered from whatever came back. A graph can **check** first:

```mermaid
flowchart LR
  S([START]) --> retrieve --> grade_documents
  grade_documents -- relevant --> answer
  grade_documents -- none, retry left --> rewrite --> retrieve
  grade_documents -- none, no retry --> answer
  answer --> E([END])
```

The three "smart" steps are plain functions you can swap: offline stand-ins now (free, deterministic), a model in the 💸 cell.

- **Grader:** keeps a chunk only if it actually answers the question (offline: Module 06's `extractive_answer` on that one chunk).
- **Rewriter:** rephrases the question (offline: a small table of rewrite rules).
- **Answerer:** a cited answer or an honest "I don't know".

In [ ]:
# 🧪 Offline: state and the offline stand-ins.
import re
from typing import Annotated, Callable, TypedDict
import operator

from langchain_core.documents import Document

from mastery.rag.citations import extractive_answer, format_citations, is_dont_know


class RagState(TypedDict, total=False):
    question: str
    query: str
    docs: list[Document]
    relevant: list[Document]
    rewrites: int
    answer: str
    trail: Annotated[list[str], operator.add]


def offline_grader(query: str, doc: Document) -> bool:
    return not is_dont_know(extractive_answer(query, [doc]))


REWRITE_RULES = [   # what an LLM rewriter would do for these phrasings
    (r"how far can (the .+?) (send data|reach)\??$", r"What is the radio range of \1?"),
    (r"how much (is|does) (the .+?)( cost)?\??$", r"What is the price of \2?"),
]


def offline_rewriter(query: str) -> str:
    for pattern, template in REWRITE_RULES:
        if re.search(pattern, query, flags=re.IGNORECASE):
            return re.sub(pattern, template, query, flags=re.IGNORECASE)
    return query


def offline_answerer(query: str, docs: list[Document]) -> str:
    return format_citations(extractive_answer(query, docs))

In [ ]:
# 🧪 Offline: the graph. Grader/rewriter/answerer are parameters, so the same graph runs live later.
from langgraph.graph import END, START, StateGraph

NOT_FOUND = "I don't know: nothing relevant in the docs."


def build_rag_graph(grader: Callable, rewriter: Callable, answerer: Callable, k: int = 4, max_rewrites: int = 1):
    def retrieve(state: RagState) -> dict:
        query = state.get("query") or state["question"]
        return {"query": query, "docs": bm25.search(query, k=k), "trail": [f"retrieve: {query}"]}

    def grade_documents(state: RagState) -> dict:
        keep = [d for d in state["docs"] if grader(state["query"], d)]
        return {"relevant": keep, "trail": [f"grade_documents: {len(keep)}/{len(state['docs'])} relevant"]}

    def rewrite(state: RagState) -> dict:
        new = rewriter(state["query"])
        return {"query": new, "rewrites": state.get("rewrites", 0) + 1, "trail": [f"rewrite: {new}"]}

    def answer(state: RagState) -> dict:
        if not state["relevant"]:
            return {"answer": NOT_FOUND, "trail": ["answer: abstain"]}
        return {"answer": answerer(state["query"], state["relevant"]), "trail": ["answer"]}

    def after_grading(state: RagState) -> str:
        if state["relevant"] or state.get("rewrites", 0) >= max_rewrites:
            return "answer"
        return "rewrite"

    builder = StateGraph(RagState)
    for node in (retrieve, grade_documents, rewrite, answer):
        builder.add_node(node)
    builder.add_edge(START, "retrieve")
    builder.add_edge("retrieve", "grade_documents")
    builder.add_conditional_edges("grade_documents", after_grading, ["answer", "rewrite"])
    builder.add_edge("rewrite", "retrieve")
    builder.add_edge("answer", END)
    return builder.compile(name="docs_rag")


rag_graph = build_rag_graph(offline_grader, offline_rewriter, offline_answerer)

for question in ["What does error E12 mean on the Nimbus Pro?",
                 "How far can the Nimbus Agro send data?",
                 "What is the capital of France?"]:
    out = rag_graph.invoke({"question": question})
    print("Q:", question)
    for step in out["trail"]:
        print("   ", step)
    print("   →", out["answer"].splitlines()[0], "\n")

**You should now see** three different paths:

1. **E12:** `retrieve` → `grade_documents: 1/4 relevant` → `answer`: "Error E12 means the rain gauge is blocked: clean the funnel."
2. **"How far ... send data":** `0/4 relevant` → `rewrite: What is the radio range of the Nimbus Agro?` → `retrieve` → `2/4 relevant` → a sentence from `nimbus-agro.md` (the offline reader copies the best-matching sentence; the model in Part 4 writes a real answer).
3. **France:** `0/0` (BM25 found no chunk sharing a word) → the rewriter has nothing better (same query) → `0/0` again → `answer: abstain`. **Bounded**: one rewrite, then stop.

### What just happened?

Grading is the step a plain chain can't do: it turns "the retriever returned *something*" into "the retriever returned *evidence*". The rewrite loop has a counter in state (`rewrites`) and a router that reads it, exactly the bounded-loop pattern from 07.03.

> 🌐 Draws the graph as a PNG through mermaid.ink (a public web service; no key, no tokens). Offline or blocked? `show_graph` prints a ⚠️ and shows the Mermaid text instead.

In [ ]:
# 🌐 Network (PNG) or offline Mermaid text.
_ = show_graph(rag_graph)

---
## Part 2 · A SQL graph (read-only)

The question "how much did I spend on restaurants?" isn't in any document: it's in a table. The steps:

| Node | Does | Model? |
|---|---|---|
| `list_tables` | Which tables exist | no |
| `get_schema` | `CREATE TABLE` + 3 sample rows of the relevant tables | no |
| `generate_query` | Writes SQL from question + schema (+ the last error) | **yes** |
| `check_query` | One statement, `SELECT`/`WITH` only, adds `LIMIT` | no |
| `run_query` | Runs on a **read-only** connection; errors come back as text | no |
| `answer` | Turns rows into a sentence | **yes** (offline: rows as text) |

Only two of six steps need a model. Everything else is ordinary, testable code from `mastery.graphs.sql`.

In [ ]:
# 🧪 Offline: Module 01's expenses CSV → a SQLite file (re-created each run; ~10 KB).
from mastery.graphs.sql import check_sql, list_tables, run_readonly_query, sqlite_from_csv, table_schema

OUT = REPO_ROOT / "data" / "outputs" / "m07"
DB = sqlite_from_csv(REPO_ROOT / "data" / "samples" / "expenses" / "expenses-2026.csv", OUT / "expenses.sqlite", "expenses")
print(list_tables(DB))
print(table_schema(DB, ["expenses"]))
print(run_readonly_query(DB, "DELETE FROM expenses"))

**You should now see** `['expenses']`, the `CREATE TABLE expenses (...)` line with three sample rows, and `Error: only SELECT queries are allowed`.

Two locks, not one: `check_sql` rejects anything but a single `SELECT`, **and** `run_readonly_query` opens the file with SQLite's `mode=ro` plus an authorizer that only allows reads. Even a query that slips past the text check can't write.

In [ ]:
# 🧪 Offline: the SQL graph. `generator(question, schema, error) -> sql` is the only model-shaped step.
class SqlState(TypedDict, total=False):
    question: str
    tables: list[str]
    schema: str
    sql: str
    error: str
    attempts: int
    rows: str
    answer: str
    trail: Annotated[list[str], operator.add]


def build_sql_graph(db_path, generator: Callable, answerer: Callable, max_attempts: int = 2):
    def list_tables_node(state: SqlState) -> dict:
        tables = list_tables(db_path)
        return {"tables": tables, "attempts": 0, "trail": [f"list_tables: {tables}"]}

    def get_schema(state: SqlState) -> dict:
        return {"schema": table_schema(db_path, state["tables"]), "trail": ["get_schema"]}

    def generate_query(state: SqlState) -> dict:
        sql = generator(state["question"], state["schema"], state.get("error", ""))
        return {"sql": sql, "attempts": state["attempts"] + 1, "trail": [f"generate_query: {sql}"]}

    def check_query(state: SqlState) -> dict:
        ok, checked = check_sql(state["sql"])
        if ok:
            return {"sql": checked, "error": "", "trail": ["check_query: ok"]}
        return {"error": checked, "trail": [f"check_query: REJECTED ({checked})"]}

    def run_query(state: SqlState) -> dict:
        result = run_readonly_query(db_path, state["sql"])
        if result.startswith("Error:"):
            return {"error": result, "trail": [f"run_query: {result}"]}
        return {"rows": result, "trail": [f"run_query: {len(result.splitlines()) - 1} row(s)"]}

    def answer(state: SqlState) -> dict:
        if state.get("error"):
            return {"answer": f"I couldn't answer that safely ({state['error']}).", "trail": ["answer: give up"]}
        return {"answer": answerer(state["question"], state["rows"]), "trail": ["answer"]}

    def retry_or_answer(state: SqlState) -> str:
        if state.get("error") and "only" not in state["error"] and state["attempts"] < max_attempts:
            return "generate_query"     # a fixable error (wrong column...): try again with the message
        return "answer"                 # success, a refused (non-SELECT) query, or out of attempts

    def after_check(state: SqlState) -> str:
        return "run_query" if not state.get("error") else retry_or_answer(state)

    builder = StateGraph(SqlState)
    builder.add_node("list_tables", list_tables_node)
    for node in (get_schema, generate_query, check_query, run_query, answer):
        builder.add_node(node)
    builder.add_edge(START, "list_tables")
    builder.add_edge("list_tables", "get_schema")
    builder.add_edge("get_schema", "generate_query")
    builder.add_edge("generate_query", "check_query")
    builder.add_conditional_edges("check_query", after_check, ["run_query", "generate_query", "answer"])
    builder.add_conditional_edges("run_query", retry_or_answer, ["generate_query", "answer"])
    builder.add_edge("answer", END)
    return builder.compile(name="expenses_sql")

In [ ]:
# 🧪 Offline: a scripted "model" (fixed SQL per question; the second question is wrong on the first try).
SCRIPTED_SQL = {
    "How much did I spend on restaurants?": ["SELECT SUM(amount) AS total FROM expenses WHERE category = 'restaurants'"],
    "Which 3 merchants got the most money?": [
        "SELECT merchant, SUM(cost) AS total FROM expenses GROUP BY merchant ORDER BY total DESC LIMIT 3",   # no `cost` column
        "SELECT merchant, SUM(amount) AS total FROM expenses GROUP BY merchant ORDER BY total DESC LIMIT 3",
    ],
    "Delete all my expenses.": ["DELETE FROM expenses"],
}


def offline_generator(question: str, schema: str, error: str) -> str:
    attempts = SCRIPTED_SQL[question]
    return attempts[1] if error and len(attempts) > 1 else attempts[0]


def offline_sql_answerer(question: str, rows: str) -> str:
    return "Rows: " + rows.replace("\n", " / ")


sql_graph = build_sql_graph(DB, offline_generator, offline_sql_answerer)
for question in SCRIPTED_SQL:
    out = sql_graph.invoke({"question": question})
    print("Q:", question)
    for step in out["trail"]:
        print("   ", step)
    print("   →", out["answer"], "\n")

**You should now see**:

1. **Restaurants:** one pass, `run_query: 1 row(s)` → `Rows: total / 1135100.0` (COP).
2. **Top merchants:** `run_query: Error: no such column: cost` → back to `generate_query` with the error → the fixed query → 3 rows, rent first.
3. **Delete:** `check_query: REJECTED (only SELECT queries are allowed)` → `answer: give up`. No retry: a refusal isn't a typo.

### What just happened?

The error **message** goes back into state, and the generator sees it on the next try: that's how a real model fixes `no such column`. `attempts` bounds the loop (2 tries).

> 🌐 Draws the graph as a PNG through mermaid.ink (a public web service; no key, no tokens). Offline or blocked? `show_graph` prints a ⚠️ and shows the Mermaid text instead.

In [ ]:
# 🌐 Network (PNG) or offline Mermaid text.
_ = show_graph(sql_graph)

---
## Part 3 · One assistant, two subgraphs (`xray=True`)

A compiled graph is a node. If parent and child share keys (`question`, `answer`), you can add the child **directly**: the child reads `question` and its `answer` flows back. `xray=True` draws the inside of each subgraph.

In [ ]:
# 🧪 Offline: a router in front of both graphs.
class AskState(TypedDict, total=False):
    question: str
    answer: str


SQL_WORDS = ("spend", "spent", "merchant", "expense", "money")


def route_question(state: AskState) -> str:
    return "expenses_sql" if any(w in state["question"].lower() for w in SQL_WORDS) else "docs_rag"


ask = StateGraph(AskState)
ask.add_node("docs_rag", rag_graph)
ask.add_node("expenses_sql", sql_graph)
ask.add_conditional_edges(START, route_question, ["docs_rag", "expenses_sql"])
ask.add_edge("docs_rag", END)
ask.add_edge("expenses_sql", END)
assistant = ask.compile(name="ask_orbita_or_my_expenses")

for question in ["What does error E12 mean on the Nimbus Pro?", "How much did I spend on restaurants?"]:
    print(route_question({"question": question}), "→", assistant.invoke({"question": question})["answer"].splitlines()[0])

**You should now see** `docs_rag → Error E12 means...` and `expenses_sql → Rows: total / 1135100.0`. The parent only keeps `question` and `answer`; each subgraph's other keys stay inside it.

> 🌐 Draws the graph as a PNG through mermaid.ink (a public web service; no key, no tokens). Offline or blocked? `show_graph` prints a ⚠️ and shows the Mermaid text instead.

In [ ]:
# 🌐 Network (PNG) or offline Mermaid text: compare the two drawings.
_ = show_graph(assistant)               # two boxes
_ = show_graph(assistant, xray=True)    # every node of both subgraphs

**You should now see** first a 4-box diagram (`START`, `docs_rag`, `expenses_sql`, `END`), then the expanded one with all ten inner nodes grouped by subgraph.

---
## Part 4 · Live: a model grades, rewrites, writes SQL

> 💸 Calls `gpt-oss-20b` a few times (one per model node), a few thousand tokens of your free-tier quota. Without `GROQ_API_KEY` it prints `⏭️ Skipped`. About 10 calls in total.

In [ ]:
# 💸 Live: swap the offline stand-ins for gpt-oss-20b. Same graphs.
from pydantic import BaseModel, Field

from mastery.rag.citations import CitedAnswer, format_context

if HAVE_GROQ_KEY:
    from mastery.models import get_model

    llm = get_model("fast", temperature=0, reasoning_effort="low")

    class Grade(BaseModel):
        relevant: bool = Field(description="Does the passage contain the answer to the question?")

    class Sql(BaseModel):
        sql: str = Field(description="One SQLite SELECT statement, no comments")

    grade_model, answer_model, sql_model = (llm.with_structured_output(Grade), llm.with_structured_output(CitedAnswer),
                                            llm.with_structured_output(Sql))

    def live_grader(query, doc):
        return grade_model.invoke(f"Question: {query}\n\nPassage:\n{doc.page_content}").relevant

    def live_rewriter(query):
        return llm.invoke(f"Rewrite this question for a keyword search over product docs. Reply with the question only.\n\n{query}").content.strip()

    def live_answerer(query, docs):
        prompt = f"Answer only from the sources; cite them. If they don't say, say you don't know.\n\n{format_context(docs)}\n\nQuestion: {query}"
        return format_citations(answer_model.invoke(prompt))

    def live_generator(question, schema, error):
        fix = f"\nThe previous query failed: {error}. Fix it." if error else ""
        return sql_model.invoke(f"SQLite schema:\n{schema}\n\nWrite one SELECT for: {question}{fix}").sql

    def live_sql_answerer(question, rows):
        return llm.invoke(f"Answer in one sentence (amounts are COP).\nQuestion: {question}\nRows:\n{rows}").content.strip()

    live_rag = build_rag_graph(live_grader, live_rewriter, live_answerer)
    live_sql = build_sql_graph(DB, live_generator, live_sql_answerer)
    out = live_rag.invoke({"question": "How far can the Nimbus Agro send data?"})
    print(out["trail"], "\n", out["answer"], "\n")
    out = live_sql.invoke({"question": "Which month had the highest total spending?"})
    print(out["trail"], "\n", out["answer"])
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** (with a key) a trail for each graph and real answers: the Agro's radio range with a citation, and a month (`2026-08`) with a total. Exact wording varies; the **trail** shows which path the model took. A live grader often accepts the first retrieval, so you may see no `rewrite` step at all.

---
## ⚠️ Pitfalls and troubleshooting (Windows Git Bash included)

| Symptom | Cause | Fix |
|---|---|---|
| `sqlite3.OperationalError: unable to open database file` | Path with non-ASCII characters or the file open in another program (DB Browser) | Close the other program; the notebook re-creates `data/outputs/m07/expenses.sqlite` |
| `PermissionError` deleting `expenses.sqlite` | A kernel still holds it | Restart the kernel, then delete |
| Amounts look wrong (`1135100.0`) | COP has no cents; SQLite returns REAL | Format in Python: `f"{{value:,.0f}} COP"` |

## ⚠️ Production notes

> ⚠️ **Production note: never let a model write to your database.**
> Use a read-only DB user (Postgres: `GRANT SELECT` only) **and** a statement check, a row limit and a timeout. The docs' SQL tutorial says the same: "check the query before executing".

> ⚠️ **Production note: grading costs a call per chunk.**
> Grade 4 chunks with a small model, or grade the whole batch in one call. Measure with LangSmith before adding steps.

## 🏋️ Exercises

1. **(Easy)** Ask `rag_graph` "How much is the Nimbus Pro?". Read the trail: which rule fired, and why is the answer still "I don't know"?
2. **(Medium)** Give the SQL graph a `max_attempts=1` and rerun "Which 3 merchants...". What's the answer, and why?
3. **(Medium)** Add a `"travel"` question to `SCRIPTED_SQL` whose first query uses `GROUP BY merchant` without an aggregate. Does SQLite reject it? What does that teach about relying on errors?
4. **(Stretch)** Make `route_question` a model call with structured output (`Literal["docs_rag", "expenses_sql"]`). Test it on 6 questions of your own.

<details>
<summary><b>Show solutions</b></summary>

**1.** The second rule already rewrites it to "What is the price of the Nimbus Pro?", yet the trail ends in `answer: abstain`, even with `k=8`. Why: the price line lives in the `## Specifications` chunk of `nimbus-pro.md`, which never says "Nimbus Pro" (the title is in another chunk), so the grader can't confirm it's about the Pro. The fix is in the **data**, not the graph: prepend the document title to each chunk (Module 06's contextual chunk headers) and rebuild `bm25`. The trail told you which node to look at.

**2.** `answer: give up` with `no such column: cost`: one attempt, no retry left.

**3.** SQLite **accepts** it (it picks an arbitrary row per group; Postgres would reject it). Errors catch only some mistakes: evaluate SQL answers against known results (Module 10).

**4.** `class Route(BaseModel): target: Literal["docs_rag", "expenses_sql"]`; `llm.with_structured_output(Route).invoke(question).target`.
</details>

## ✅ Checkpoint

- [ ] My RAG graph grades, rewrites at most once, and abstains when nothing is relevant.
- [ ] My SQL graph refuses writes, fixes a wrong column using the error, and stops after 2 attempts.
- [ ] I composed both under a router and read the `xray=True` diagram.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Open the docs' [Custom RAG agent](https://docs.langchain.com/oss/python/langgraph/agentic-rag): its first node lets the model **decide** whether to retrieve at all (`generate_query_or_respond` with a retriever tool). Ours always retrieves. Write in [`docs/research-log.md`](../../docs/research-log.md) which design you'd use for the Orbita support bot and why (cost per question, predictability, chit-chat).

## Summary

- Each step a node: retrieval, grading, rewriting and answering become separately visible and testable.
- Loops are bounded by a counter in state (`rewrites`, `attempts`) and a router.
- SQL from a model: check the text, run read-only, feed errors back, cap the attempts.
- Compiled graphs are nodes; `xray=True` shows their insides.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: the **module project**: read [project/README.md](project/README.md) (spec, modes, acceptance criteria), then open [`07-project-content-pipeline.ipynb`](project/07-project-content-pipeline.ipynb).

[Module 07 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)